In [26]:
import warnings
import os

# Suppress Python warnings
warnings.filterwarnings('ignore')

# Suppress TensorFlow, standard error, and system-level logs
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

In [27]:
import os
import random
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import torchvision
from torchvision import transforms
from torchvision.models import (
    convnext_tiny,
    ConvNeXt_Tiny_Weights
)

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from tqdm.auto import tqdm

# CONFIG

ROOT = Path(
    "/kaggle/input/datasets/samasiayushman/"
    "small-model-track/Training/Training/data/IR"
)

SEED = 42

IMG_SIZE = 224
BATCH_SIZE = 64
NUM_WORKERS = 2

EPOCHS = 15
LR = 2e-4
WEIGHT_DECAY = 1e-4

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


def seed_everything(seed=42):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


seed_everything(SEED)

PyTorch: 2.10.0+cu128
Torchvision: 0.25.0+cu128
Device: cuda
GPU: Tesla T4


In [28]:
from pathlib import Path
import pandas as pd
import re

ROOT = Path(
    "/kaggle/input/datasets/samasiayushman/"
    "small-model-track/Training/Training/data/IR"
)

rows = []

for class_dir in sorted(ROOT.iterdir()):

    if not class_dir.is_dir():
        continue

    class_name = class_dir.name

    for user_dir in class_dir.iterdir():

        if not user_dir.is_dir():
            continue

        user = user_dir.name

        for seq_dir in user_dir.iterdir():

            if not seq_dir.is_dir():
                continue

            files = sorted(seq_dir.glob("*.png"))

            if not files:
                continue

            # Extract frame numbers
            frame_numbers = []

            for f in files:
                m = re.search(r'_(\d+)\.png$', f.name)

                if m:
                    frame_numbers.append(int(m.group(1)))

            rows.append({
                "class": class_name,
                "user": user,
                "sequence": seq_dir.name,
                "path": str(seq_dir),
                "n_frames": len(files),
                "first_frame": min(frame_numbers)
                    if frame_numbers else None,
                "last_frame": max(frame_numbers)
                    if frame_numbers else None
            })

df = pd.DataFrame(rows)

print("Sequences:", len(df))
print("Columns:", df.columns.tolist())

print("\nFirst rows:")
display(df.head(10))

print("\nFrames:")
print(df["n_frames"].describe())

print("\nSequences per user:")
print(df.groupby("user").size())

print("\nSequences per class:")
print(
    df.groupby("class")
      .size()
      .sort_values()
)

Sequences: 2933
Columns: ['class', 'user', 'sequence', 'path', 'n_frames', 'first_frame', 'last_frame']

First rows:


,class,user,sequence,path,n_frames,first_frame,last_frame
0,0_Wash_face,user21,4-2-3,/kaggle/input/datasets/samasiayushman/small-mo...,9,24,32
1,0_Wash_face,user21,1-1-3,/kaggle/input/datasets/samasiayushman/small-mo...,5,44,48
2,0_Wash_face,user21,1-1-2,/kaggle/input/datasets/samasiayushman/small-mo...,10,70,79
3,0_Wash_face,user21,1-1-1,/kaggle/input/datasets/samasiayushman/small-mo...,14,75,88
4,0_Wash_face,user21,4-2-1,/kaggle/input/datasets/samasiayushman/small-mo...,7,51,57
5,0_Wash_face,user21,4-2-2,/kaggle/input/datasets/samasiayushman/small-mo...,6,31,36
6,0_Wash_face,user6,1-1-3,/kaggle/input/datasets/samasiayushman/small-mo...,34,69,102
7,0_Wash_face,user6,1-1-2,/kaggle/input/datasets/samasiayushman/small-mo...,45,127,171
8,0_Wash_face,user6,1-1-1,/kaggle/input/datasets/samasiayushman/small-mo...,47,136,182
9,0_Wash_face,user18,7-1-1,/kaggle/input/datasets/samasiayushman/small-mo...,20,62,81



Frames:
count    2933.000000
mean       29.293556
std        21.894297
min         1.000000
25%        14.000000
50%        24.000000
75%        38.000000
max       236.000000
Name: n_frames, dtype: float64

Sequences per user:
user
user1     149
user16    186
user17    166
user18    178
user19    186
user2     167
user20    159
user21    133
user22    192
user23    132
user24    159
user3     161
user4     134
user5     100
user6     201
user7     184
user8     165
user9     181
dtype: int64

Sequences per class:
class
25_Watch_TV                            12
16_Fold_clothes                        24
35_Do_lunges                           26
33_Lie_down                            33
14_Wipe_bowls                          35
28_Jog_in_place                        37
18_Write                               38
26_Play_games                          40
27_Take_a_selfie                       41
3_Take_off_clothes                     41
19_Make_a_phone_call                   43
0_Wash_face

In [29]:
# CLASS MAPPING
# =========================================================

classes = sorted([
    x.name for x in ROOT.iterdir()
    if x.is_dir()
])

class_to_idx = {
    cls: i for i, cls in enumerate(classes)
}

idx_to_class = {
    i: cls for cls, i in class_to_idx.items()
}

print("Classes:", len(classes))


# =========================================================
# FRAME DATAFRAME
# =========================================================

rows = []

for _, r in tqdm(
    df.iterrows(),
    total=len(df),
    desc="Building frame index"
):

    class_name = r["class"]
    user = r["user"]
    sequence = r["sequence"]

    seq_path = Path(r["path"])

    for img_path in seq_path.glob("*.png"):

        rows.append({
            "path": str(img_path),
            "class": class_name,
            "label": class_to_idx[class_name],
            "user": user,
            "sequence": sequence,
        })

frames_df = pd.DataFrame(rows)

print("\nTotal frames:", len(frames_df))
display(frames_df.head())

Classes: 40


Building frame index:   0%|          | 0/2933 [00:00<?, ?it/s]


Total frames: 85918


,path,class,label,user,sequence
0,/kaggle/input/datasets/samasiayushman/small-mo...,0_Wash_face,0,user21,4-2-3
1,/kaggle/input/datasets/samasiayushman/small-mo...,0_Wash_face,0,user21,4-2-3
2,/kaggle/input/datasets/samasiayushman/small-mo...,0_Wash_face,0,user21,4-2-3
3,/kaggle/input/datasets/samasiayushman/small-mo...,0_Wash_face,0,user21,4-2-3
4,/kaggle/input/datasets/samasiayushman/small-mo...,0_Wash_face,0,user21,4-2-3


In [30]:
VAL_USERS = {"user24", "user4", "user5"}

train_df = frames_df[
    ~frames_df["user"].isin(VAL_USERS)
].reset_index(drop=True)

val_df = frames_df[
    frames_df["user"].isin(VAL_USERS)
].reset_index(drop=True)

print("Train frames:", len(train_df))
print("Val frames  :", len(val_df))

print("\nTrain users:")
print(sorted(train_df["user"].unique()))

print("\nVal users:")
print(sorted(val_df["user"].unique()))

print("\nTrain class distribution:")
print(train_df["class"].value_counts().sort_index())

print("\nValidation class distribution:")
print(val_df["class"].value_counts().sort_index())

Train frames: 72519
Val frames  : 13399

Train users:
['user1', 'user16', 'user17', 'user18', 'user19', 'user2', 'user20', 'user21', 'user22', 'user23', 'user3', 'user6', 'user7', 'user8', 'user9']

Val users:
['user24', 'user4', 'user5']

Train class distribution:
class
0_Wash_face                            722
10_Stir_drinks                        2411
11_Peel_fruits                        2684
12_Sweep_the_floor                    3375
13_Mop_the_floor                      2086
14_Wipe_bowls                         1292
15_Wipe_windows_and_tables            1827
16_Fold_clothes                       1248
17_Tap_the_keyboard                   2783
18_Write                               962
19_Make_a_phone_call                  1087
1_Brush_teeth                         1021
20_Check_the_time                     1307
21_Read_documents                     2427
22_Turn_pages                         1625
23_Listen_to_music_with_headphones    2669
24_Use_a_mobile_phone                 10

In [31]:
def sample_sequence_frames(group, max_frames=12):

    group = group.sort_values("path")

    if len(group) <= max_frames:
        return group

    indices = np.linspace(
        0,
        len(group) - 1,
        max_frames
    ).astype(int)

    return group.iloc[indices]


train_sampled = (
    train_df
    .groupby(
        ["class", "user", "sequence"],
        group_keys=False
    )
    .apply(
        lambda x: sample_sequence_frames(
            x,
            max_frames=12
        )
    )
    .reset_index(drop=True)
)

print("Original training frames:", len(train_df))
print("Sampled training frames :", len(train_sampled))

print(
    "Reduction:",
    f"{100 * (1 - len(train_sampled)/len(train_df)):.1f}%"
)

Original training frames: 72519
Sampled training frames : 28708
Reduction: 60.4%


In [32]:
train_transform = transforms.Compose([
    transforms.Resize(
        (IMG_SIZE, IMG_SIZE)
    ),

    transforms.RandomResizedCrop(
        IMG_SIZE,
        scale=(0.80, 1.0),
        ratio=(0.85, 1.15)
    ),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.RandomAffine(
        degrees=8,
        translate=(0.05, 0.05),
        scale=(0.95, 1.05)
    ),

    transforms.ToTensor(),

    # Convert grayscale -> 3 channels
    transforms.Lambda(
        lambda x: x.repeat(3, 1, 1)
    ),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


val_transform = transforms.Compose([
    transforms.Resize(
        (IMG_SIZE, IMG_SIZE)
    ),

    transforms.ToTensor(),

    transforms.Lambda(
        lambda x: x.repeat(3, 1, 1)
    ),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [33]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
from tqdm.auto import tqdm

In [37]:
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset
from PIL import Image
from pathlib import Path


SEQ_LEN = 24
IMG_W = 160
IMG_H = 120


class IRSequenceDatasetV2(Dataset):

    def __init__(
        self,
        dataframe,
        class_to_idx,
        seq_len=24,
        training=False
    ):
        self.df = dataframe.reset_index(drop=True)
        self.class_to_idx = class_to_idx
        self.seq_len = seq_len
        self.training = training

    def __len__(self):
        return len(self.df)

    def _frame_indices(self, n):

        if n >= self.seq_len:

            if self.training:

                max_start = n - self.seq_len

                if max_start > 0:

                    start = np.random.randint(
                        0,
                        max_start + 1
                    )

                    return np.linspace(
                        start,
                        n - 1,
                        self.seq_len
                    ).astype(np.int64)

            return np.linspace(
                0,
                n - 1,
                self.seq_len
            ).astype(np.int64)

        # Short sequence:
        # repeat/interpolate existing frames
        return np.linspace(
            0,
            n - 1,
            self.seq_len
        ).astype(np.int64)

    def _normalize(self, img):

        # VERY IMPORTANT: float32
        img = np.asarray(
            img,
            dtype=np.float32
        )

        p2, p98 = np.percentile(
            img,
            [2, 98]
        )

        # Force scalar float32
        p2 = np.float32(p2)
        p98 = np.float32(p98)

        if p98 > p2:

            img = np.clip(
                img,
                p2,
                p98
            ).astype(np.float32)

            img = (
                img - p2
            ) / (
                p98 - p2 + np.float32(1e-6)
            )

        else:

            img = (
                img / np.float32(255.0)
            ).astype(np.float32)

        return img.astype(np.float32)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        files = list(
            Path(row["path"]).glob("*.png")
        )

        # Extract actual frame number
        def frame_number(p):

            try:
                return int(
                    p.stem.split("_")[-1]
                )

            except:
                return 0

        files = sorted(
            files,
            key=frame_number
        )

        indices = self._frame_indices(
            len(files)
        )

        raw_frames = []

        for i in indices:

            img = np.asarray(
                Image.open(
                    files[i]
                ).convert("L"),
                dtype=np.float32
            )

            img = self._normalize(img)

            img = cv2.resize(
                img,
                (IMG_W, IMG_H),
                interpolation=cv2.INTER_AREA
            )

            # Make absolutely sure
            img = np.asarray(
                img,
                dtype=np.float32
            )

            raw_frames.append(img)

        raw = np.stack(
            raw_frames
        ).astype(np.float32)

        # =================================================
        # SPATIAL GRADIENT
        # =================================================

        gradients = []

        for img in raw:

            # Explicit float32 input
            img = np.ascontiguousarray(
                img,
                dtype=np.float32
            )

            gx = cv2.Sobel(
                img,
                cv2.CV_32F,
                1,
                0,
                ksize=3
            )

            gy = cv2.Sobel(
                img,
                cv2.CV_32F,
                0,
                1,
                ksize=3
            )

            magnitude = np.sqrt(
                gx * gx +
                gy * gy
            ).astype(np.float32)

            magnitude /= (
                magnitude.max()
                + np.float32(1e-6)
            )

            gradients.append(
                magnitude
            )

        gradients = np.stack(
            gradients
        ).astype(np.float32)

        # =================================================
        # TEMPORAL DIFFERENCE
        # =================================================

        differences = np.zeros_like(
            raw,
            dtype=np.float32
        )

        differences[1:] = np.abs(
            raw[1:] - raw[:-1]
        ).astype(np.float32)

        # =================================================
        # 3 CHANNEL INPUT
        # =================================================

        frames = np.stack(
            [
                raw,
                gradients,
                differences
            ],
            axis=-1
        ).astype(np.float32)

        # =================================================
        # AUGMENTATION
        # =================================================

        if self.training:

            if np.random.rand() < 0.5:

                frames = frames[
                    :, :, ::-1, :
                ].copy()

            if np.random.rand() < 0.3:

                factor = np.float32(
                    np.random.uniform(
                        0.90,
                        1.10
                    )
                )

                frames[:, :, :, 0] *= factor

                frames[:, :, :, 0] = np.clip(
                    frames[:, :, :, 0],
                    0,
                    1
                )

        # =================================================
        # T,H,W,C → T,C,H,W
        # =================================================

        frames = torch.from_numpy(
            frames.copy()
        ).permute(
            0,
            3,
            1,
            2
        ).float()

        label = int(
            self.class_to_idx[
                row["class"]
            ]
        )

        return frames, label

In [38]:
VAL_USERS = {
    "user24",
    "user4",
    "user5"
}

train_seq_df = df[
    ~df["user"].isin(VAL_USERS)
].reset_index(drop=True)

val_seq_df = df[
    df["user"].isin(VAL_USERS)
].reset_index(drop=True)

print("Train sequences:", len(train_seq_df))
print("Val sequences  :", len(val_seq_df))

Train sequences: 2540
Val sequences  : 393


In [39]:
train_dataset = IRSequenceDatasetV2(
    train_seq_df,
    class_to_idx,
    seq_len=24,
    training=True
)

val_dataset = IRSequenceDatasetV2(
    val_seq_df,
    class_to_idx,
    seq_len=24,
    training=False
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [40]:
x, y = train_dataset[0]

print("Single sequence:", x.shape)
print("dtype:", x.dtype)
print("min:", x.min().item())
print("max:", x.max().item())

print("\nChannels:")
print("IR       :", x[:, 0].min().item(),
      x[:, 0].max().item())

print("Gradient :", x[:, 1].min().item(),
      x[:, 1].max().item())

print("Difference:", x[:, 2].min().item(),
      x[:, 2].max().item())

Single sequence: torch.Size([24, 3, 120, 160])
dtype: torch.float32
min: 0.0
max: 1.0

Channels:
IR       : 0.0 1.0
Gradient : 0.0 0.9999997615814209
Difference: 0.0 0.7303286790847778


In [41]:
xb, yb = next(iter(train_loader))

print("Batch:", xb.shape)
print("Labels:", yb.shape)
print("dtype:", xb.dtype)
print("min/max:", xb.min().item(), xb.max().item())

Batch: torch.Size([32, 24, 3, 120, 160])
Labels: torch.Size([32])
dtype: torch.float32
min/max: 0.0 1.0


In [43]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

CUDA available: True
GPU count: 2
0 Tesla T4
1 Tesla T4


In [44]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# Residual Block
# ============================================================

class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_ch, out_ch,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False
        )
        self.bn1 = nn.BatchNorm2d(out_ch)

        self.conv2 = nn.Conv2d(
            out_ch, out_ch,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )
        self.bn2 = nn.BatchNorm2d(out_ch)

        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_ch, out_ch,
                    kernel_size=1,
                    stride=stride,
                    bias=False
                ),
                nn.BatchNorm2d(out_ch)
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):

        identity = self.shortcut(x)

        x = self.conv1(x)
        x = self.bn1(x)
        x = F.gelu(x)

        x = self.conv2(x)
        x = self.bn2(x)

        x = x + identity
        x = F.gelu(x)

        return x


# ============================================================
# Custom CNN Encoder
# ============================================================

class IRFrameEncoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(
                3, 32,
                kernel_size=5,
                stride=2,
                padding=2,
                bias=False
            ),
            nn.BatchNorm2d(32),
            nn.GELU()
        )

        self.layer1 = nn.Sequential(
            ResidualBlock(32, 32),
            ResidualBlock(32, 32)
        )

        self.layer2 = nn.Sequential(
            ResidualBlock(32, 64, stride=2),
            ResidualBlock(64, 64)
        )

        self.layer3 = nn.Sequential(
            ResidualBlock(64, 128, stride=2),
            ResidualBlock(128, 128)
        )

        self.layer4 = nn.Sequential(
            ResidualBlock(128, 192, stride=2),
            ResidualBlock(192, 192)
        )

        self.layer5 = nn.Sequential(
            ResidualBlock(192, 256, stride=2),
            ResidualBlock(256, 256)
        )

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.projection = nn.Sequential(
            nn.Linear(256, 384),
            nn.LayerNorm(384),
            nn.GELU(),
            nn.Dropout(0.15)
        )

    def forward(self, x):

        x = self.stem(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.layer5(x)

        x = self.pool(x)

        x = torch.flatten(x, 1)

        x = self.projection(x)

        return x


# ============================================================
# Temporal Attention
# ============================================================

class TemporalAttention(nn.Module):

    def __init__(self, dim):
        super().__init__()

        self.score = nn.Sequential(
            nn.Linear(dim, dim // 2),
            nn.Tanh(),
            nn.Linear(dim // 2, 1)
        )

    def forward(self, x):

        # x: [B, T, D]

        scores = self.score(x)

        weights = torch.softmax(scores, dim=1)

        context = torch.sum(
            x * weights,
            dim=1
        )

        return context, weights


# ============================================================
# Full IR V2 Model
# ============================================================

class IRSequenceModelV2(nn.Module):

    def __init__(
        self,
        num_classes=40,
        lstm_hidden=256,
        lstm_layers=2
    ):
        super().__init__()

        self.encoder = IRFrameEncoder()

        self.lstm = nn.LSTM(
            input_size=384,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.25 if lstm_layers > 1 else 0.0
        )

        temporal_dim = lstm_hidden * 2

        self.attention = TemporalAttention(
            temporal_dim
        )

        self.classifier = nn.Sequential(
            nn.Linear(temporal_dim, 384),
            nn.LayerNorm(384),
            nn.GELU(),
            nn.Dropout(0.30),

            nn.Linear(384, 192),
            nn.GELU(),
            nn.Dropout(0.20),

            nn.Linear(192, num_classes)
        )

    def forward(self, x):

        # x = [B, T, C, H, W]

        B, T, C, H, W = x.shape

        # Process all frames through CNN simultaneously
        x = x.reshape(
            B * T,
            C,
            H,
            W
        )

        x = self.encoder(x)

        # [B*T, 384]
        x = x.reshape(
            B,
            T,
            384
        )

        # Temporal modeling
        x, _ = self.lstm(x)

        # Attention over frames
        x, attention_weights = self.attention(x)

        # Classification
        logits = self.classifier(x)

        return logits, attention_weights

In [46]:
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = IRSequenceModelV2(
    num_classes=len(classes),
    lstm_hidden=256,
    lstm_layers=2
)

model = model.to(DEVICE)


# Use both T4 GPUs
if torch.cuda.device_count() >= 2:
    print("Using", torch.cuda.device_count(), "GPUs")
    model = nn.DataParallel(model)

print(model)
def count_parameters(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


params = count_parameters(model)

print(f"Parameters: {params:,}")
print(f"FP32 size: {params * 4 / 1024**2:.2f} MB")

Using 2 GPUs
DataParallel(
  (module): IRSequenceModelV2(
    (encoder): IRFrameEncoder(
      (stem): Sequential(
        (0): Conv2d(3, 32, kernel_size=(5, 5), stride=(2, 2), padding=(2, 2), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): GELU(approximate='none')
      )
      (layer1): Sequential(
        (0): ResidualBlock(
          (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (shortcut): Identity()
        )
        (1): ResidualBlock(
          (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(32, eps=1e-05, 

In [47]:
xb, yb = next(iter(train_loader))

xb = xb.to(DEVICE)

with torch.no_grad():

    logits, attn = model(xb)

print("Input :", xb.shape)
print("Logits:", logits.shape)
print("Attention:", attn.shape)

Input : torch.Size([32, 24, 3, 120, 160])
Logits: torch.Size([32, 40])
Attention: torch.Size([32, 24, 1])


In [48]:
from torch.utils.data import WeightedRandomSampler, DataLoader
import numpy as np
import torch

# ------------------------------------------------------------
# Compute class weights from TRAINING sequences only
# ------------------------------------------------------------

train_labels = train_seq_df["class"].map(class_to_idx).values

class_counts = np.bincount(
    train_labels,
    minlength=len(classes)
)

# Softer than inverse-frequency:
# prevents tiny classes from dominating
class_weights = 1.0 / np.sqrt(class_counts)

# Normalize
class_weights = class_weights / class_weights.mean()

print("Class counts:")
for i, c in enumerate(classes):
    print(f"{i:2d} {c:35s} {class_counts[i]:4d}  weight={class_weights[i]:.3f}")


# Weight assigned to each sequence
sample_weights = class_weights[train_labels]

sample_weights = torch.as_tensor(
    sample_weights,
    dtype=torch.double
)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)


# ------------------------------------------------------------
# Recreate loaders
# ------------------------------------------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    sampler=sampler,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))

Class counts:
 0 0_Wash_face                           32  weight=1.211
 1 10_Stir_drinks                       106  weight=0.665
 2 11_Peel_fruits                        79  weight=0.771
 3 12_Sweep_the_floor                    57  weight=0.907
 4 13_Mop_the_floor                      47  weight=0.999
 5 14_Wipe_bowls                         32  weight=1.211
 6 15_Wipe_windows_and_tables            49  weight=0.979
 7 16_Fold_clothes                       21  weight=1.495
 8 17_Tap_the_keyboard                   71  weight=0.813
 9 18_Write                              30  weight=1.251
10 19_Make_a_phone_call                  35  weight=1.158
11 1_Brush_teeth                         42  weight=1.057
12 20_Check_the_time                     85  weight=0.743
13 21_Read_documents                     64  weight=0.856
14 22_Turn_pages                         47  weight=0.999
15 23_Listen_to_music_with_headphones    69  weight=0.825
16 24_Use_a_mobile_phone                 36  weight=1.142


In [51]:
criterion = nn.CrossEntropyLoss(
    label_smoothing=0.05
)
import math

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

NUM_EPOCHS = 35
WARMUP_EPOCHS = 3

def lr_lambda(epoch):

    if epoch < WARMUP_EPOCHS:
        return float(epoch + 1) / WARMUP_EPOCHS

    progress = (
        epoch - WARMUP_EPOCHS
    ) / (
        NUM_EPOCHS - WARMUP_EPOCHS
    )

    return 0.5 * (
        1.0 + math.cos(math.pi * progress)
    )

scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda
)
scaler = torch.cuda.amp.GradScaler(
    enabled=torch.cuda.is_available()
)

In [52]:
from collections import Counter

def train_one_epoch(model, loader, criterion, optimizer, scaler):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for xb, yb in loader:

        xb = xb.to(
            DEVICE,
            non_blocking=True
        )

        yb = yb.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.cuda.amp.autocast(
            enabled=torch.cuda.is_available()
        ):

            logits, _ = model(xb)

            loss = criterion(
                logits,
                yb
            )

        scaler.scale(loss).backward()

        # Gradient clipping
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        scaler.step(optimizer)
        scaler.update()

        running_loss += (
            loss.item() * xb.size(0)
        )

        preds = logits.argmax(dim=1)

        correct += (
            (preds == yb).sum().item()
        )

        total += yb.size(0)

    return (
        running_loss / total,
        correct / total
    )


@torch.no_grad()
def validate(model, loader, criterion):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    for xb, yb in loader:

        xb = xb.to(
            DEVICE,
            non_blocking=True
        )

        yb = yb.to(
            DEVICE,
            non_blocking=True
        )

        with torch.cuda.amp.autocast(
            enabled=torch.cuda.is_available()
        ):

            logits, _ = model(xb)

            loss = criterion(
                logits,
                yb
            )

        running_loss += (
            loss.item() * xb.size(0)
        )

        preds = logits.argmax(dim=1)

        correct += (
            (preds == yb).sum().item()
        )

        total += yb.size(0)

        all_preds.extend(
            preds.cpu().numpy()
        )

        all_labels.extend(
            yb.cpu().numpy()
        )

    return (
        running_loss / total,
        correct / total,
        np.array(all_labels),
        np.array(all_preds)
    )

In [ ]:
import time
import copy

best_val_acc = 0.0
best_state = None
patience = 8
epochs_without_improvement = 0

history = []

for epoch in range(NUM_EPOCHS):

    start_time = time.time()

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        scaler
    )

    val_loss, val_acc, y_true, y_pred = validate(
        model,
        val_loader,
        criterion
    )

    scheduler.step()

    lr = optimizer.param_groups[0]["lr"]

    elapsed = time.time() - start_time

    print(
        f"Epoch {epoch+1:02d} | "
        f"Train Loss {train_loss:.4f} | "
        f"Train Acc {train_acc:.4f} | "
        f"Val Loss {val_loss:.4f} | "
        f"Val Acc {val_acc:.4f} | "
        f"LR {lr:.2e} | "
        f"{elapsed:.1f}s"
    )

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "lr": lr
    })

    # --------------------------------------------------------
    # Save best model
    # --------------------------------------------------------

    if val_acc > best_val_acc:

        best_val_acc = val_acc
        epochs_without_improvement = 0

        best_state = copy.deepcopy(
            model.module.state_dict()
            if isinstance(model, nn.DataParallel)
            else model.state_dict()
        )

        torch.save(
            best_state,
            "/kaggle/working/ir_v2_best.pth"
        )

        print(
            f"  ★ NEW BEST: {best_val_acc:.4f}"
        )

    else:
        epochs_without_improvement += 1

    # --------------------------------------------------------
    # Early stopping
    # --------------------------------------------------------

    if epochs_without_improvement >= patience:

        print(
            f"\nEarly stopping at epoch {epoch+1}"
        )

        break


print(
    f"\nBest Validation Accuracy: "
    f"{best_val_acc:.4f}"
)

Epoch 01 | Train Loss 3.5364 | Train Acc 0.0752 | Val Loss 3.4127 | Val Acc 0.1196 | LR 2.00e-04 | 271.3s
  ★ NEW BEST: 0.1196
Epoch 02 | Train Loss 3.2280 | Train Acc 0.1126 | Val Loss 3.1946 | Val Acc 0.0687 | LR 3.00e-04 | 245.0s
Epoch 03 | Train Loss 3.0351 | Train Acc 0.1343 | Val Loss 2.9945 | Val Acc 0.1603 | LR 3.00e-04 | 246.3s
  ★ NEW BEST: 0.1603
Epoch 04 | Train Loss 2.8411 | Train Acc 0.1772 | Val Loss 2.7587 | Val Acc 0.1908 | LR 2.99e-04 | 239.6s
  ★ NEW BEST: 0.1908
Epoch 05 | Train Loss 2.7266 | Train Acc 0.2071 | Val Loss 2.9016 | Val Acc 0.1883 | LR 2.97e-04 | 236.0s
Epoch 06 | Train Loss 2.5743 | Train Acc 0.2374 | Val Loss 2.8975 | Val Acc 0.1705 | LR 2.94e-04 | 237.6s
Epoch 07 | Train Loss 2.4909 | Train Acc 0.2614 | Val Loss 2.4802 | Val Acc 0.2545 | LR 2.89e-04 | 235.9s
  ★ NEW BEST: 0.2545
Epoch 08 | Train Loss 2.3822 | Train Acc 0.2945 | Val Loss 2.6233 | Val Acc 0.2061 | LR 2.82e-04 | 235.4s
Epoch 09 | Train Loss 2.2781 | Train Acc 0.3291 | Val Loss 2.3943 | 